In [2]:
import numpy as np

# Define hidden states and observed symbols
hidden_states = ['E', 'I']
observed_symbols = ['A', 'C', 'G', 'T']

# Transition probabilities between states
transition_probabilities = {
    'E': {'E': 0.9, 'I': 0.1},
    'I': {'E': 0.1, 'I': 0.9}
}

# Emission probabilities from states to observed symbols
emission_probabilities = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

# Initial probabilities of starting in a given state
initial_probabilities = {'E': 0.5, 'I': 0.5}

def compute_log_prob_of_path(state_sequence, observed_sequence):
    if len(state_sequence) != len(observed_sequence):
        raise ValueError("The state sequence and observed sequence must have the same length.")

    log_probability = np.log(initial_probabilities[state_sequence[0]]) + np.log(emission_probabilities[state_sequence[0]][observed_sequence[0]])
    for i in range(1, len(observed_sequence)):
        previous_state = state_sequence[i - 1]
        current_state = state_sequence[i]
        trans_prob = transition_probabilities[previous_state][current_state]
        emit_prob = emission_probabilities[current_state][observed_sequence[i]]
        log_probability += np.log(trans_prob) + np.log(emit_prob)
    return round(log_probability, 2)

def viterbi_algorithm(observed_sequence):
    if len(observed_sequence) == 0:
        return (0.0, [])

    dp_table = [{}]  # Viterbi matrix
    backtrace_paths = {}

    # Initialization step
    for state in hidden_states:
        dp_table[0][state] = np.log(initial_probabilities[state]) + np.log(emission_probabilities[state][observed_sequence[0]])
        backtrace_paths[state] = [state]

    # Recursion step
    for t in range(1, len(observed_sequence)):
        dp_table.append({})
        new_paths = {}

        for current_state in hidden_states:
            max_prob, best_prev_state = max(
                (dp_table[t - 1][prev_state] +
                 np.log(transition_probabilities[prev_state][current_state]) +
                 np.log(emission_probabilities[current_state][observed_sequence[t]]), prev_state)
                for prev_state in hidden_states
            )
            dp_table[t][current_state] = max_prob
            new_paths[current_state] = backtrace_paths[best_prev_state] + [current_state]

        backtrace_paths = new_paths

    # Termination step
    final_time_step = len(observed_sequence) - 1
    best_final_prob, best_final_state = max((dp_table[final_time_step][state], state) for state in hidden_states)
    return (best_final_prob, backtrace_paths[best_final_state])

# Example usage
example_state_sequence = "EEEEEEEEEEEEEEEEEEIIIIIII"
example_observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Align lengths
sequence_length = min(len(example_state_sequence), len(example_observed_sequence))
example_state_sequence = example_state_sequence[:sequence_length]
example_observed_sequence = example_observed_sequence[:sequence_length]

log_prob_result = compute_log_prob_of_path(example_state_sequence, example_observed_sequence)
print(f"Log-probability of given path: {log_prob_result}")

viterbi_log_prob, viterbi_optimal_path = viterbi_algorithm(example_observed_sequence)
print(f"Viterbi best log-prob: {round(viterbi_log_prob, 2)}")
print(f"Viterbi best path: {''.join(viterbi_optimal_path)}")


Log-probability of given path: -40.95
Viterbi best log-prob: -37.88
Viterbi best path: EEEEEEEEEEEEEEEEEEEEEEEEE
